In [ ]:
from sqlalchemy import create_engine

db_params = {
    'host': 'localhost',
    'database': 'postgres',
    'user': 'postgres',
    'password': 'postgres',
    'port': 5432
}

conn = create_engine(f"postgresql://{db_params['user']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/{db_params['database']}")

In [ ]:
import pandas as pd

# Query to get mutation status counts by project and stage
query = """
SELECT 
    p.id AS project_id,
    p.root_path,
    pmr.step,
    pmr.stage,
    pmr.variant,
    pmr.status,
    COUNT(*) AS mutation_count
FROM 
    pit_mutation_report pmr
JOIN 
    project p ON pmr.project_id = p.id
GROUP BY 
    p.id, p.root_path, pmr.step, pmr.stage, pmr.variant, pmr.status
ORDER BY 
    p.id, pmr.step, pmr.stage, pmr.variant, pmr.status
"""

# Query to get detected mutation counts
detected_query = """
SELECT 
    p.id AS project_id,
    pmr.step,
    pmr.stage,
    pmr.variant,
    SUM(pmr.is_detected::int) AS detected_count,
    COUNT(*) AS total_count
FROM 
    pit_mutation_report pmr
JOIN 
    project p ON pmr.project_id = p.id
GROUP BY 
    p.id, pmr.step, pmr.stage, pmr.variant
"""

# Execute queries and load results into DataFrames
mutation_stats = pd.read_sql_query(query, conn)
detected_stats = pd.read_sql_query(detected_query, conn)

# Function to create display variant names
def get_display_variant(row):
    if row['stage'] == 'COLLECT_PIT_DATA_ORIGINAL' and pd.isna(row['variant']):
        return 'ORIGINAL'
    elif row['stage'] == 'COLLECT_PIT_DATA_INITIAL' and pd.isna(row['variant']):
        return 'INITIAL'
    else:
        return row['variant']

# Create display variant columns
mutation_stats['display_variant'] = mutation_stats.apply(get_display_variant, axis=1)
detected_stats['display_variant'] = detected_stats.apply(get_display_variant, axis=1)

# Calculate detection percentage with the new name
detected_stats['DETECTED OF TOTAL %'] = (detected_stats['detected_count'] / detected_stats['total_count'] * 100).round(2)

# Create a pivot table for mutation status counts
pivot_table = mutation_stats.pivot_table(
    index=['project_id', 'root_path', 'step', 'stage', 'display_variant'],
    columns='status',
    values='mutation_count',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Calculate total mutations
pivot_table['TOTAL'] = pivot_table.select_dtypes(include=['int64']).sum(axis=1)

# Merge the detected percentage and count into the pivot table
pivot_table = pd.merge(
    pivot_table,
    detected_stats[['project_id', 'step', 'stage', 'display_variant', 'DETECTED OF TOTAL %', 'detected_count']],
    on=['project_id', 'step', 'stage', 'display_variant'],
    how='left'
)

# Add DETECTED column (using the detected_count from the query)
pivot_table['DETECTED'] = pivot_table['detected_count']

variant_order = {
    'ORIGINAL': 0, 
    'INITIAL': 1, 
    'BASELINE': 2, 
    'IMPROVED_5_TRIES': 3, 
    'IMPROVED_20_TRIES': 4
}

# Add sorting columns
pivot_table['variant_order'] = pivot_table['display_variant'].map(variant_order)

# Sort the DataFrame
pivot_table = pivot_table.sort_values(['root_path', 'project_id', 'step', 'variant_order'])

# Create a dictionary to store INITIAL detection percentages for each project
initial_detection = {}
for _, row in pivot_table[pivot_table['display_variant'] == 'INITIAL'].iterrows():
    initial_detection[row['project_id']] = row['DETECTED OF TOTAL %']

# Add improvement column
def calculate_improvement(row):
    if row['display_variant'] in ['ORIGINAL', 'INITIAL']:
        return None  # No improvement value for ORIGINAL and INITIAL
    initial = initial_detection.get(row['project_id'])
    if initial is None:
        return None  # No INITIAL variant to compare with
    return round(row['DETECTED OF TOTAL %'] - initial, 2)

# Apply the function to create the new column
pivot_table['D.O.T. IMPROVEMENT'] = pivot_table.apply(calculate_improvement, axis=1)

# Calculate covered mutations (all mutations except NO_COVERAGE)
if 'NO_COVERAGE' in pivot_table.columns:
    pivot_table['COVERAGE'] = pivot_table['TOTAL'] - pivot_table['NO_COVERAGE']
else:
    pivot_table['COVERAGE'] = pivot_table['TOTAL']  # If NO_COVERAGE column doesn't exist

# Calculate detected among covered percentage
pivot_table['DETECTED OF COVERED %'] = (
    pivot_table['detected_count'] / pivot_table['COVERAGE'] * 100
).round(2)

# Calculate improvement for detected among covered
initial_covered_detection = {}
for _, row in pivot_table[pivot_table['display_variant'] == 'INITIAL'].iterrows():
    initial_covered_detection[row['project_id']] = row['DETECTED OF COVERED %']

def calculate_covered_improvement(row):
    if row['display_variant'] in ['ORIGINAL', 'INITIAL']:
        return None
    initial = initial_covered_detection.get(row['project_id'])
    if initial is None:
        return None
    return round(row['DETECTED OF COVERED %'] - initial, 2)

pivot_table['D.O.C. IMPROVEMENT'] = pivot_table.apply(calculate_covered_improvement, axis=1)

# Extract project name from root_path for cleaner display
pivot_table['project_name'] = pivot_table['root_path'].apply(lambda x: x.split('/')[-1])

# Define the order of columns
ordered_columns = ['project_id', 'project_name', 'stage', 'display_variant', 'TOTAL']
ordered_columns.extend(['TOTAL', 'NO_COVERAGE', 'COVERAGE', 'SURVIVED', 'DETECTED', 'KILLED', 'TIMED_OUT', 'RUN_ERROR'])
ordered_columns.extend(['DETECTED OF TOTAL %', 'D.O.T. IMPROVEMENT', 'DETECTED OF COVERED %', 'D.O.C. IMPROVEMENT'])

# Remove detected_count as it's only used for calculations
pivot_table = pivot_table.drop(columns=['detected_count'], errors='ignore')

# Select only columns that exist in the DataFrame
final_columns = [col for col in ordered_columns if col in pivot_table.columns]

# Create the final result table
result_table = pivot_table[final_columns].rename(columns={'display_variant': 'variant'})

# Display the results
print("Mutation Status Counts by Project, Stage, and Variant with Improvement Analysis")
display(result_table)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.gridspec import GridSpec

# Query to get mutation results by mutator type
mutator_query = """
SELECT 
    p.id AS project_id,
    p.root_path,
    pmr.stage,
    pmr.variant,
    pmr.mutator,
    pmr.is_detected,
    COUNT(*) AS mutation_count
FROM 
    pit_mutation_report pmr
JOIN 
    project p ON pmr.project_id = p.id
GROUP BY 
    p.id, p.root_path, pmr.stage, pmr.variant, pmr.mutator, pmr.is_detected
ORDER BY 
    p.id, pmr.stage, pmr.variant, pmr.mutator
"""

# Execute query and load results into DataFrame
mutator_stats = pd.read_sql_query(mutator_query, conn)

# Function to create display variant names
def get_display_variant(row):
    if row['stage'] == 'COLLECT_PIT_DATA_ORIGINAL' and pd.isna(row['variant']):
        return 'ORIGINAL'
    elif row['stage'] == 'COLLECT_PIT_DATA_INITIAL' and pd.isna(row['variant']):
        return 'INITIAL'
    else:
        return row['variant']

# Create display variant column
mutator_stats['display_variant'] = mutator_stats.apply(get_display_variant, axis=1)

# Extract project name from root_path
mutator_stats['project_name'] = mutator_stats['root_path'].apply(lambda x: x.split('/')[-1])

# Extract base project name
def get_base_project_name(project_name):
    # Match everything before "-es-" followed by any characters
    match = re.match(r'^(.*?)(?=-es-)', project_name)
    if match:
        return match.group(1)
    else:
        # Fallback: return the original name if pattern doesn't match
        return project_name

# Add base project name column
mutator_stats['base_project_name'] = mutator_stats['project_name'].apply(get_base_project_name)

# Function to simplify mutator names (extract class name only)
def simplify_mutator_name(mutator):
    # Extract the class name from the fully qualified name
    match = re.search(r'\.([^.]+)$', mutator)
    if match:
        return match.group(1)
    return mutator

# Apply the function to create a simplified mutator column
mutator_stats['simple_mutator'] = mutator_stats['mutator'].apply(simplify_mutator_name)

# Create a pivot table for mutator analysis
mutator_pivot = mutator_stats.pivot_table(
    index=['project_id', 'project_name', 'base_project_name', 'stage', 'display_variant', 'simple_mutator'],
    columns='is_detected',
    values='mutation_count',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Rename columns for clarity
mutator_pivot = mutator_pivot.rename(columns={0: 'not_detected', 1: 'detected'})

# Calculate total mutations and detection rate for each mutator
mutator_pivot['total'] = mutator_pivot['not_detected'] + mutator_pivot['detected']
mutator_pivot['detection_rate'] = (mutator_pivot['detected'] / mutator_pivot['total'] * 100).round(2)

# Function to determine variant order based on the specified rules
def get_variant_order(variant):
    if variant == 'ORIGINAL':
        return 0
    elif variant == 'INITIAL':
        return 1
    elif variant == 'BASELINE':
        return 2
    elif variant and variant.startswith('NAIVE_'):
        # Extract the number of tries from NAIVE_{number}_TRIES
        match = re.search(r'NAIVE_(\d+)_TRIES', variant)
        if match:
            return 3 + int(match.group(1)) / 1000  # Use decimal to maintain order
        return 3
    elif variant and variant.startswith('IMPROVED_'):
        # Extract the number of tries from IMPROVED_{number}_TRIES
        match = re.search(r'IMPROVED_(\d+)_TRIES', variant)
        if match:
            return 4 + int(match.group(1)) / 1000  # Use decimal to maintain order
        return 4
    else:
        return 999  # Other variants at the end

# Define stage order
stage_order = {'COLLECT_PIT_DATA_ORIGINAL': 0, 'COLLECT_PIT_DATA_INITIAL': 1, 'COLLECT_PIT_DATA_GENERALIZED': 2}

# Add sorting columns
mutator_pivot['stage_order'] = mutator_pivot['stage'].map(stage_order)
mutator_pivot['variant_order'] = mutator_pivot['display_variant'].apply(get_variant_order)

# Sort the DataFrame
mutator_pivot = mutator_pivot.sort_values(['base_project_name', 'project_id', 'stage_order', 'variant_order', 'simple_mutator'])

# Create a dictionary to store INITIAL detection rates for each project and mutator
initial_detection_by_mutator = {}
for _, row in mutator_pivot[mutator_pivot['display_variant'] == 'INITIAL'].iterrows():
    key = (row['project_id'], row['simple_mutator'])
    initial_detection_by_mutator[key] = row['detection_rate']

# Add improvement column
def calculate_mutator_improvement(row):
    if row['display_variant'] in ['ORIGINAL', 'INITIAL']:
        return None  # No improvement value for ORIGINAL and INITIAL
    key = (row['project_id'], row['simple_mutator'])
    initial = initial_detection_by_mutator.get(key)
    if initial is None:
        return None  # No INITIAL variant to compare with
    return round(row['detection_rate'] - initial, 2)

# Apply the function to create the new column
mutator_pivot['improvement'] = mutator_pivot.apply(calculate_mutator_improvement, axis=1)

# Clean up the DataFrame
result_columns = ['project_id', 'project_name', 'base_project_name', 'stage', 'display_variant', 'simple_mutator', 
                 'detected', 'not_detected', 'total', 'detection_rate', 'improvement']
mutator_results = mutator_pivot[result_columns].rename(columns={'display_variant': 'variant', 'simple_mutator': 'mutator'})

# Get all unique variants and sort them by our custom order
all_variants = sorted(mutator_results['variant'].unique(), key=get_variant_order)

# Define a colorblind-friendly color palette with more colors for all variants
base_colors = [
    '#1f77b4',  # blue - ORIGINAL
    '#ff7f0e',  # orange - INITIAL
    '#2ca02c',  # green - BASELINE
    '#d62728',  # red 
    '#9467bd',  # purple
    '#8c564b',  # brown
    '#e377c2',  # pink
    '#7f7f7f',  # gray
    '#bcbd22',  # olive
    '#17becf',  # cyan
    '#aec7e8',  # light blue
    '#ffbb78',  # light orange
    '#98df8a',  # light green
    '#ff9896',  # light red
    '#c5b0d5',  # light purple
]

# Generate a color map for all variants
variant_colors = {}
naive_colors = plt.cm.Reds(np.linspace(0.3, 0.8, 10))  # For NAIVE variants
improved_colors = plt.cm.Blues(np.linspace(0.3, 0.8, 10))  # For IMPROVED variants

for i, variant in enumerate(all_variants):
    if variant == 'ORIGINAL':
        variant_colors[variant] = base_colors[0]
    elif variant == 'INITIAL':
        variant_colors[variant] = base_colors[1]
    elif variant == 'BASELINE':
        variant_colors[variant] = base_colors[2]
    elif variant and variant.startswith('NAIVE_'):
        # Extract the number of tries
        match = re.search(r'NAIVE_(\d+)_TRIES', variant)
        if match:
            tries = min(int(match.group(1)), 100) // 10  # Scale to fit our color array
            variant_colors[variant] = naive_colors[tries-1]
        else:
            variant_colors[variant] = base_colors[3 % len(base_colors)]
    elif variant and variant.startswith('IMPROVED_'):
        # Extract the number of tries
        match = re.search(r'IMPROVED_(\d+)_TRIES', variant)
        if match:
            tries = min(int(match.group(1)), 100) // 10  # Scale to fit our color array
            variant_colors[variant] = improved_colors[tries-1]
        else:
            variant_colors[variant] = base_colors[4 % len(base_colors)]
    else:
        variant_colors[variant] = base_colors[(i+5) % len(base_colors)]

# Define which variants are improvement variants (exclude ORIGINAL, INITIAL, BASELINE)
improvement_variants = [v for v in all_variants if v not in ['ORIGINAL', 'INITIAL', 'BASELINE']]

# Calculate average improvement by mutator type
improvement_by_mutator = mutator_results[mutator_results['improvement'].notna()].groupby('mutator')['improvement'].mean().sort_values(ascending=False)

# Create a bar chart of average improvement by mutator
plt.figure(figsize=(12, 6))
bars = improvement_by_mutator.plot(kind='bar', color='#2ca02c')  # Use green for improvement bars
plt.title('Average Improvement in Detection Rate by Mutator Type')
plt.xlabel('Mutator')
plt.ylabel('Average Improvement (%)')
plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Get unique project IDs
project_ids = mutator_results['project_id'].unique()
project_count = len(project_ids)

# Get all unique mutators across all projects to ensure consistent x-axis
all_mutators = sorted(mutator_results['mutator'].unique())
mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}

# Create a figure with custom grid layout (detection rate plots wider than improvement plots)
fig = plt.figure(figsize=(18, 4 * project_count))  # Increased width to accommodate more variants
gs = GridSpec(project_count, 2, width_ratios=[3, 2])

# Set fixed x-axis limits for all plots
x_min = -0.5
x_max = len(all_mutators) - 0.5

# Set up consistent y-axis limits
detection_y_max = 0
improvement_y_max = 0
improvement_y_min = 0

# First pass to determine y-axis limits
for project_id in project_ids:
    project_data = mutator_results[mutator_results['project_id'] == project_id]
    detection_y_max = max(detection_y_max, project_data['detection_rate'].max() * 1.1)

    project_improvements = project_data[project_data['improvement'].notna()]['improvement']
    if not project_improvements.empty:
        improvement_y_max = max(improvement_y_max, project_improvements.max() * 1.1)
        improvement_y_min = min(improvement_y_min, project_improvements.min() * 1.1)

# Create legend handles for all variants
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=variant_colors[variant]) for variant in all_variants]
legend_labels = all_variants

# Add a legend at the top of the figure
fig.legend(
    legend_handles, 
    legend_labels, 
    loc='upper center', 
    ncol=min(len(all_variants), 5),
    bbox_to_anchor=(0.5, 0.98),
    fontsize='small'
)

# Create plots for each project ID
for i, project_id in enumerate(project_ids):
    # Filter data for this project
    project_data = mutator_results[mutator_results['project_id'] == project_id]

    # Skip if no data for this project ID
    if project_data.empty:
        continue

    project_name = project_data['project_name'].iloc[0]
    base_name = project_data['base_project_name'].iloc[0]

    # Get the axes for this project
    ax1 = fig.add_subplot(gs[i, 0])
    ax2 = fig.add_subplot(gs[i, 1])

    # 1. Detection Rate Chart
    # Calculate the center position for each group of bars
    width = 0.8 / len(all_variants)  # Adjust width based on number of variants
    num_variants = len(all_variants)
    total_width = width * num_variants
    group_offsets = -total_width/2 + width/2  # Start offset to center the group

    # Plot each variant as a group of bars
    for variant_idx, variant in enumerate(all_variants):
        variant_data = project_data[project_data['variant'] == variant]

        # Create a dictionary mapping mutator to detection rate for this variant
        detection_rates = {row['mutator']: row['detection_rate'] for _, row in variant_data.iterrows()}

        # For each mutator position, plot the bar if data exists
        for mutator, pos in mutator_positions.items():
            rate = detection_rates.get(mutator, 0)
            if rate > 0:  # Only plot non-zero values
                offset = group_offsets + width * variant_idx
                ax1.bar(pos + offset, rate, width, color=variant_colors[variant])

    # Add labels and title
    ax1.set_ylabel('Detection Rate (%)')
    ax1.set_title(f'Detection Rate - Project ID: {project_id} - {project_name}')

    # Set fixed axis limits
    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(0, detection_y_max)

    # Only add x-tick labels for the last project
    if i == project_count - 1:
        ax1.set_xticks(np.arange(len(all_mutators)))
        ax1.set_xticklabels(all_mutators, rotation=90, ha='center')
    else:
        ax1.set_xticks(np.arange(len(all_mutators)))
        ax1.set_xticklabels([])

    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    # 2. Improvement Chart
    width = 0.8 / len(improvement_variants) if improvement_variants else 0.4  # Adjust width based on number of variants
    num_improvement_variants = len(improvement_variants)

    if num_improvement_variants > 0:
        # Calculate the center position for each group of bars
        total_width = width * num_improvement_variants
        group_offsets = -total_width/2 + width/2  # Start offset to center the group

        # Plot each variant as a group of bars (excluding ORIGINAL, INITIAL, and BASELINE)
        for variant_idx, variant in enumerate(improvement_variants):
            variant_data = project_data[project_data['variant'] == variant]

            # Create a dictionary mapping mutator to improvement for this variant
            improvements = {row['mutator']: row['improvement'] for _, row in variant_data.iterrows() if row['improvement'] is not None}

            # For each mutator position, plot the bar if data exists
            for mutator, pos in mutator_positions.items():
                impr = improvements.get(mutator, 0)
                if impr != 0:  # Only plot non-zero values
                    offset = group_offsets + width * variant_idx
                    ax2.bar(pos + offset, impr, width, color=variant_colors[variant])

    # Add labels and title
    ax2.set_ylabel('Improvement (%)')
    ax2.set_title(f'Improvement - Project ID: {project_id} - {project_name}')

    # Set fixed axis limits
    ax2.set_xlim(x_min, x_max)
    ax2.set_ylim(improvement_y_min, improvement_y_max)

    # Only add x-tick labels for the last project
    if i == project_count - 1:
        ax2.set_xticks(np.arange(len(all_mutators)))
        ax2.set_xticklabels(all_mutators, rotation=90, ha='center')
    else:
        ax2.set_xticks(np.arange(len(all_mutators)))
        ax2.set_xticklabels([])

    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    ax2.axhline(y=0, color='r', linestyle='-', alpha=0.3)

# Adjust layout with more space at the top for legends
plt.tight_layout(rect=[0, 0.03, 1, 0.92])
plt.subplots_adjust(hspace=0.3, top=0.88)
plt.show()
